# Circuit Tracing — Gemma-3-4B-IT

Generate a full rollout from Gemma-3-4B-IT, then use **circuit-tracer** to build
an attribution graph for the first generated token (the answer).

In [ ]:
# ── 0. HuggingFace auth (required for gated Gemma models) ───────────────────────
from huggingface_hub import login
login()  # will prompt for token, or set HF_TOKEN env var beforehand

In [ ]:
# ── 1. Full rollout with Gemma-3-4B-IT ──────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "google/gemma-3-4b-it"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map=device, torch_dtype=torch.bfloat16
)
model.eval()

# The prompt forces the answer to appear as the very first generated token
PROMPT = (
    "I have a dog that is 3 years old. How old she will be in 4 years? "
    "Reply with answer in <answer> tag, and later a logical explanation."
)
chat = [{"role": "user", "content": PROMPT}]

# Apply chat template and append the start of the answer tag so the model
# generates the number directly as the first token
prompt_text = tokenizer.apply_chat_template(
    chat, tokenize=False, add_generation_prompt=True
) + "<answer>"

print("Prompt (with forced <answer> prefix):")
print(prompt_text)

inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(device)
prompt_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    output_ids = model.generate(
        **inputs, max_new_tokens=512, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

full_response = tokenizer.decode(output_ids[0])
generated_only = tokenizer.decode(output_ids[0][prompt_len:])
first_gen_token = tokenizer.decode(output_ids[0][prompt_len])

print(f"\nFirst generated token: '{first_gen_token}'")
print(f"\nFull generation:\n{generated_only}")

In [ ]:
# ── 2. Free the HF model — circuit-tracer loads its own copy ────────────────────
del model
torch.cuda.empty_cache()

In [ ]:
# ── 3. Write a local transcoder config for Gemma-3-4B-IT ────────────────────────
# circuit-tracer needs a config.yaml pointing to the transcoder weights.
# google/gemma-scope-2-4b-it has transcoders at layers 9, 17, 22, 29.
# We use the 262k-width, medium-L0, non-affine variant.
import yaml, os

TC_CONFIG_PATH = "/tmp/gemma3_4b_transcoder_config.yaml"

config = {
    "model_name": "google/gemma-3-4b-it",
    "model_kind": "transcoder_set",
    "feature_input_hook": "ln2.hook_normalized",
    "feature_output_hook": "hook_mlp_out",
    "repo_id": "google/gemma-scope-2-4b-it",
    "transcoders": [
        f"hf://google/gemma-scope-2-4b-it/transcoder/layer_{layer}_width_262k_l0_medium/params.safetensors"
        for layer in [9, 17, 22, 29]
    ],
}

with open(TC_CONFIG_PATH, "w") as f:
    yaml.dump(config, f)

print(f"Wrote transcoder config to {TC_CONFIG_PATH}")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# ── 4. Load ReplacementModel via circuit-tracer (nnsight backend) ────────────────
from circuit_tracer import ReplacementModel

replacement_model = ReplacementModel.from_pretrained(
    model_name=MODEL_NAME,
    transcoder_set=TC_CONFIG_PATH,
    backend="nnsight",
    device=torch.device(device),
    dtype=torch.bfloat16,
)
print(f"ReplacementModel loaded on {device}")

In [ ]:
# ── 5. Compute attribution graph for the first generated token ──────────────────
from circuit_tracer import attribute

# The prompt includes the forced <answer> prefix — circuit-tracer will compute
# attributions for the next-token prediction at the last position (the answer).
graph = attribute(
    prompt=prompt_text,
    model=replacement_model,
    max_n_logits=5,
    desired_logit_prob=0.95,
    verbose=True,
)

print(f"\nAttribution graph computed")
print(f"  Logit tokens: {graph.logit_tokens}")

# Save the graph
GRAPH_PATH = "../activations/circuit_graph_gemma4b_dog_age.pt"
graph.to_pt(GRAPH_PATH)
print(f"  Saved to {GRAPH_PATH}")

In [ ]:
# ── 6. Inspect the attribution graph ────────────────────────────────────────────
# Reload if needed: graph = Graph.from_pt(GRAPH_PATH)
from circuit_tracer import Graph

graph = Graph.from_pt(GRAPH_PATH)
print(f"Logit tokens: {graph.logit_tokens}")
print(f"Logit token IDs: {graph.logit_token_ids}")